# AIPV1EMI0001 — rebuild and deploy on an EdgeAI Station

Step-by-step guide to rebuild **AIPV1EMI0001** (YOLOv7-tiny `car` / `person` / `truck` detector)
from this repository, register it on an EdgeAI Station, verify its callback output, and install
its Node-RED dashboard. Run the cells top to bottom; each section ends with a check.

**Where to run this notebook:** in the station's Jupyter environment (`ms8_jupyter`, the
App-Builder container). It has `eais-tools`, the CUDA toolkit, DeepStream headers and `trtexec`,
and it shares `/data/aip` (AI processors) and `/data/projects` (Node-RED projects) with the
station services, so every step runs in place.

| Step | What happens |
|---|---|
| 0 | Check the environment and the AIP folder location |
| 1 | Download the model from Hugging Face (ONNX, optionally the prebuilt engine) |
| 2 | Build the parser and deployment engine with visible `make` / `trtexec` commands |
| 2A | Optional: compare NMS-free ONNX Runtime / TensorRT FP32 / FP16 inference |
| 3 | Validate and package with `eais-tools` |
| 4 | Register the AIP with the video-streaming service (ms3) |
| 5 | Run a test pipeline and check the callback output contract |
| 6 | Install the Node-RED flow and the UIBuilder dashboard |
| 7 | Clean up |

## 0. Prerequisites

The station only discovers AI processors under `/data/aip/<unique_id>`, so this repository must be
cloned **as `/data/aip/AIPV1EMI0001`**. From a Jupyter terminal:

```bash
cd /data/aip
git clone https://github.com/edge-ai/AIPV1EMI0001.git
```

Then open `/data/aip/AIPV1EMI0001/localdev.ipynb` (this notebook) from that folder.

If `/data/aip/AIPV1EMI0001` already exists (an earlier install), move it aside first, for example
`mv AIPV1EMI0001 AIPV1EMI0001.backup.$(date +%s)`.

The next cell checks the location and the tools. The tested environment is JetPack 6.0 /
L4T R36.3 / TensorRT 8.6.2 / DeepStream 7.0 with `eais-tools` 0.9.2.

In [ ]:
import os, re, shutil, subprocess
from pathlib import Path

def run(args, *, cwd=None, env=None, expected=(0,)):
    result = subprocess.run(args, cwd=cwd, env=env)
    if result.returncode not in expected:
        raise subprocess.CalledProcessError(result.returncode, args)
    return result

AIP_ID = "AIPV1EMI0001"
AIP_ROOT = Path.cwd()
TRTEXEC = Path(os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"))

assert AIP_ROOT == Path("/data/aip") / AIP_ID, (
    f"Run this notebook from /data/aip/{AIP_ID} (current directory: {AIP_ROOT})")
assert (AIP_ROOT / "manifest.json").is_file(), "manifest.json not found: is this the repository root?"

for tool in ("eais-tools", "make", "git", "npm", "timeout"):
    assert shutil.which(tool), f"{tool} not found in PATH"
assert Path("/usr/local/cuda/bin/nvcc").is_file(), "CUDA toolkit (nvcc) not found"
assert Path("/opt/nvidia/deepstream/deepstream/sources/includes").is_dir(), "DeepStream headers not found"
assert TRTEXEC.is_file(), f"trtexec not found at {TRTEXEC} (set TRTEXEC_BIN)"

trt_banner = subprocess.run([str(TRTEXEC), "--help"], capture_output=True, text=True).stdout
TRT_VERSION = (re.search(r"TensorRT v(\d+)", trt_banner) or [None, "unknown"])[1]
L4T = Path("/etc/nv_tegra_release").read_text().split(",")[:2] if Path("/etc/nv_tegra_release").exists() else ["unknown"]
free_gb = shutil.disk_usage(AIP_ROOT).free / 1e9

print(f"AIP folder    : {AIP_ROOT}")
print(f"TensorRT      : v{TRT_VERSION}")
print(f"L4T           : {' '.join(s.strip() for s in L4T)}")
print(f"Free disk     : {free_gb:.1f} GB")
assert free_gb > 1, "Less than 1 GB free on /data/aip"
print("OK: environment ready")

## 1. Download the model

The weights are **not stored in this repository**. They are published on Hugging Face at
[`Edgematrix-JP/AIPV1EMI0001`](https://huggingface.co/Edgematrix-JP/AIPV1EMI0001) and verified
here with SHA-256. The weights are licensed under GPL-3.0, matching the YOLOv7 training code.

Choose how to get the TensorRT engine:

- `ENGINE_SOURCE = "build"` (default) downloads the **ONNX** and builds the engine on this station
  in step 2 (allow 30–60 minutes on a busy Orin NX). The verified target is JetPack 6.0 /
  L4T R36.3 / TensorRT 8.6.2 / DeepStream 7.0; other SDK versions need separate validation.
- `ENGINE_SOURCE = "prebuilt"` downloads the published FP16 engine (dynamic batch 1–8). TensorRT
  engines only load on the TensorRT version they were built with, so this option is limited to
  **JetPack 6 / L4T R36.3 / TensorRT 8.6.2**.

`RUN_BENCHMARKS = True` also downloads the ONNX for the optional comparison in section 2A,
even when the deployment engine is prebuilt. It builds **two additional engines** and can
take another 30-60 minutes or more on a busy station. Set it to `False` for deployment only.
Run step 0 first: it defines `AIP_ROOT`, the tool paths and the checked command helper.

In [ ]:
import hashlib, urllib.request

ENGINE_SOURCE = "build"  # "build" or "prebuilt"
RUN_BENCHMARKS = True  # False skips section 2A and its two extra engine builds

HF_BASE = "https://huggingface.co/Edgematrix-JP/AIPV1EMI0001/resolve/main"
MODEL_DIR = AIP_ROOT / "models" / "AIPV1EMI0001_Yolo_Detector"
ENGINE_PATH = MODEL_DIR / "1" / "yolov7-fp16-1x8x8.engine"
PREBUILT_TRT_VERSION = "8602"

# Hugging Face file -> (destination, sha256)
ONNX = {"yolov7tiny-640.onnx": (
    MODEL_DIR / "yolov7tiny-640.onnx",
    "5796e9e0836751e3b7a9623cc6db976ac7fd087fcfb2742d20d06dcf0a0b3560")}
PREBUILT = {"yolov7-fp16-1x8x8-jp6-l4t36.3-arm64.engine": (
    ENGINE_PATH,
    "670150b495a0f120d178fbbb1e3f29758ccbee710e4477bf6847f46517a873b8")}

assert ENGINE_SOURCE in ("build", "prebuilt"), ENGINE_SOURCE
if ENGINE_SOURCE == "prebuilt":
    assert TRT_VERSION == PREBUILT_TRT_VERSION, (
        f"The prebuilt engine needs TensorRT v{PREBUILT_TRT_VERSION}, this station has v{TRT_VERSION}. "
        "Use ENGINE_SOURCE = 'build'.")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

downloads = dict(ONNX) if ENGINE_SOURCE == "build" or RUN_BENCHMARKS else {}
if ENGINE_SOURCE == "prebuilt":
    downloads.update(PREBUILT)
for name, (dest, expected) in downloads.items():
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and sha256(dest) == expected:
        print(f"OK (cached): {dest.relative_to(AIP_ROOT)}")
        continue
    print(f"Downloading {name} ...")
    urllib.request.urlretrieve(f"{HF_BASE}/{name}", dest)
    actual = sha256(dest)
    if actual != expected:
        dest.unlink()
        raise AssertionError(f"checksum mismatch for {name}: {actual}")
    print(f"OK: {dest.relative_to(AIP_ROOT)}")

## 2. Build the runtime artifacts

The commands are visible here so you can understand and adjust the build.
`bin/build.sh` remains the non-interactive alternative; do not run both.

### 2.1 Compile the DeepStream parser and build the deployment engine

`make` compiles `src/nvdsinfer_customparser/` against the installed CUDA and DeepStream
headers. This parser converts the **deployment model's NMS outputs** into DeepStream boxes.
The deployment graph keeps `EfficientNMS_TRT`; do not replace it with section 2A's raw graph.

| `trtexec` argument | Meaning and tuning guidance |
|---|---|
| `--onnx` / `--saveEngine` | Source graph and compiled, hardware/SDK-specific engine |
| `--fp16` | Allow FP16 tactics with higher-precision fallbacks; not every operation necessarily runs in FP16 |
| `--minShapes=images:1x3x640x640` | Smallest supported batch, NCHW layout |
| `--optShapes=images:8x3x640x640` | Batch TensorRT optimizes its tactic selection for |
| `--maxShapes=images:8x3x640x640` | Largest accepted batch; the Triton config and manifest also specify 8 |
| `--workspace=1024` | Limit the workspace memory pool to 1024 MiB (TensorRT 8.6 syntax); not a total process/GPU-memory cap |
| `--timingCacheFile` | Cache tactic timings to accelerate compatible later builds; not an engine cache |
| `--skipInference` | Build only; the next cell performs a separate inference smoke test |

`optShapes` does **not** force inference batch size: one engine supports batches 1 through 8.
Changing maximum batch or resolution also requires changing and validating the deployment
configs, not just these flags. Keep the 1/8/8 profile for the supplied AIP.

Higher workspace limits allow more tactics but do not guarantee a faster engine.
Jetson uses shared system/GPU memory: 4096 MiB previously caused a build to be OOM-killed
with the stack running. Start at 1024 MiB, keep other station workloads in mind, and inspect
the saved build log on errors. `--useSpinWait` is a benchmark synchronization option, not
an engine optimization, so it is not needed in a build-only command.

In [ ]:
import shlex

WORKSPACE_MIB = 1024
BUILD_TIMEOUT_SECONDS = 3600
assert isinstance(WORKSPACE_MIB, int) and WORKSPACE_MIB > 0

def run_logged(args, log_path, *, cwd=None, timeout=BUILD_TIMEOUT_SECONDS):
    print(shlex.join([str(arg) for arg in args]), flush=True)
    print(f"Log: {log_path}", flush=True)
    with open(log_path, "w") as log:
        result = subprocess.run(args, cwd=cwd, stdout=log, stderr=subprocess.STDOUT,
                                timeout=timeout)
    if result.returncode:
        print(Path(log_path).read_text()[-12000:])
        raise subprocess.CalledProcessError(result.returncode, args)
    print("OK", flush=True)

build_logs = AIP_ROOT / "tmp" / "build"
build_logs.mkdir(parents=True, exist_ok=True)
nvcc = subprocess.run(["/usr/local/cuda/bin/nvcc", "--version"],
                      capture_output=True, text=True, check=True).stdout
cuda_match = re.search(r"release (\d+\.\d+)", nvcc)
assert cuda_match, nvcc
parser_src = AIP_ROOT / "src" / "nvdsinfer_customparser"
run_logged(["make", "-C", str(parser_src), f"CUDA_VER={cuda_match[1]}"],
           build_logs / "parser.log")
shutil.copy2(parser_src / "libnvds_infercustomparser.so", MODEL_DIR)

if ENGINE_SOURCE == "build":
    ENGINE_PATH.parent.mkdir(parents=True, exist_ok=True)
    run_logged([
        str(TRTEXEC), f"--onnx={next(iter(ONNX.values()))[0]}", "--fp16",
        f"--workspace={WORKSPACE_MIB}",
        "--minShapes=images:1x3x640x640",
        "--optShapes=images:8x3x640x640",
        "--maxShapes=images:8x3x640x640",
        f"--saveEngine={ENGINE_PATH}",
        f"--timingCacheFile={AIP_ROOT / 'bin' / 'timing.cache'}", "--skipInference",
    ], build_logs / "deployment-fp16.log")
else:
    print(f"Keeping the checksum-verified prebuilt deployment engine: {ENGINE_PATH}")

In [ ]:
for path in (MODEL_DIR / "libnvds_infercustomparser.so", ENGINE_PATH):
    assert path.is_file() and path.stat().st_size > 0, f"missing build output: {path}"
    print(f"OK: {path.relative_to(AIP_ROOT)} ({path.stat().st_size / 1e6:.1f} MB)")
run([str(TRTEXEC), f"--loadEngine={ENGINE_PATH}", "--shapes=images:1x3x640x640",
     "--duration=1", "--warmUp=200", "--iterations=1"])
print("OK: engine deserializes and runs inference on this station")

## 2A. Optional: benchmark raw inference backends

**Goal:** compare ONNX Runtime and TensorRT **with the same timing method**, without a
serving framework or video pipeline. We remove the terminal `EfficientNMS_TRT` node from
a **copy** of the ONNX and expose its inputs: decoded boxes `[batch, 25200, 4]` and class
scores `[batch, 25200, 3]`. This shared graph still includes box decoding and score
multiplication, but excludes NMS. No weights are changed.

| Variant | Executed by (one Python process) |
|---|---|
| ONNX Runtime CUDA FP32 | `onnxruntime.InferenceSession` with only `CUDAExecutionProvider`, TF32 disabled |
| TensorRT FP32 | TensorRT Python runtime executing an engine built without `--fp16` and with `--noTF32` |
| TensorRT FP16 | TensorRT Python runtime executing an engine built with `--fp16 --noTF32` |

`trtexec --onnx` **builds and then runs a TensorRT engine**; it is not an ONNX Runtime
baseline. We use `trtexec` only to *build* the two engines, then execute them with the
station's own TensorRT libraries. FP16 permits FP32 fallback for some layers.
Shape/control operations may run on CPU even with the ONNX Runtime CUDA provider.

**Inference runs locally in one Python process, without a model server.**
EAIS Tools remain responsible for AIP validation, packaging and full-pipeline tests in
the following sections; deployment still uses `nvinferserver`. Nothing here restarts
services, registers benchmark models or alters the deployment graph.
Allow extra build time/RAM; do not run overlapping benchmarks on the same station.

In [ ]:
if RUN_BENCHMARKS:
    import json, uuid
    import numpy as np
    import onnx
    from datetime import datetime, timezone
    from onnx import TensorProto, helper, shape_inference

    import onnxruntime as ort
    assert "CUDAExecutionProvider" in ort.get_available_providers(), "ONNX Runtime CUDA provider missing"
    assert shutil.disk_usage(AIP_ROOT).free > 2 * 1024**3, "Need at least 2 GiB free"
    BENCH_ROOT = AIP_ROOT / "tmp" / "inference-benchmark" / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
    BENCH_ROOT.mkdir(parents=True)
    RAW_ONNX = BENCH_ROOT / "raw.onnx"
    source_onnx, expected_sha = next(iter(ONNX.values()))
    assert sha256(source_onnx) == expected_sha, "Download the original ONNX in step 1"
    graph = onnx.load(source_onnx)
    nms_nodes = [n for n in graph.graph.node if n.op_type == "EfficientNMS_TRT"]
    assert len(nms_nodes) == 1, "Expected exactly one terminal NMS node"
    nms = nms_nodes[0]
    assert set(nms.output) == {v.name for v in graph.graph.output}
    assert len(nms.input) == 2, "Expected boxes and scores"
    raw_names = list(nms.input)
    graph.graph.node.remove(nms)
    del graph.graph.output[:]
    graph.graph.output.extend([
        helper.make_tensor_value_info(name, TensorProto.FLOAT, ["batch", "candidates", width])
        for name, width in zip(raw_names, (4, 3))
    ])
    graph = shape_inference.infer_shapes(graph)
    renames = dict(zip(raw_names, ("raw_boxes", "raw_scores")))
    for node in graph.graph.node:
        for field in (node.input, node.output):
            for index, name in enumerate(field):
                field[index] = renames.get(name, name)
    for value in list(graph.graph.output) + list(graph.graph.value_info):
        value.name = renames.get(value.name, value.name)
    assert not any(n.domain == "TRT" for n in graph.graph.node)
    onnx.checker.check_model(graph)
    onnx.save(graph, RAW_ONNX)
    assert sha256(source_onnx) == expected_sha, "The deployment graph must remain unchanged"
    print(f"Benchmark-only graph: {RAW_ONNX} ({len(graph.graph.node)} nodes)")
    print("Outputs: raw_boxes [batch, candidates, 4], raw_scores [batch, candidates, 3]")
else:
    print("Skipping raw inference benchmarks (RUN_BENCHMARKS=False).")

### 2A.1 Build the two benchmark engines

Both use **min/opt/max batch = 1/8/8**, the same 640×640 input and workspace limit.
Only precision flags differ. `--noTF32` disables TensorRT's default TF32 tactics.
FP16 can improve speed and memory use, but the gain depends on the graph, batch and
hardware; measure it rather than assuming a fixed speedup.

These are **not deployment engines**: they return unsuppressed candidates and stay in
a unique `tmp/inference-benchmark/` directory, outside the packaged model directory.
The deployment graph retains NMS. Commands, build logs and results are saved for review.
Changing maximum batch or resolution also requires changing benchmark inputs/outputs.

In [ ]:
if RUN_BENCHMARKS:
    ENGINES = {
        "TensorRT FP32": BENCH_ROOT / "raw-fp32.engine",
        "TensorRT FP16": BENCH_ROOT / "raw-fp16.engine",
    }
    for label, engine in ENGINES.items():
        command = [
            str(TRTEXEC), f"--onnx={RAW_ONNX}", "--noTF32",
            f"--workspace={WORKSPACE_MIB}",
            "--minShapes=images:1x3x640x640",
            "--optShapes=images:8x3x640x640",
            "--maxShapes=images:8x3x640x640",
            f"--saveEngine={engine}",
            f"--timingCacheFile={engine.with_suffix('.cache')}", "--skipInference",
        ]
        if label == "TensorRT FP16":
            command.append("--fp16")
        run_logged(command, engine.with_suffix(".build.log"))
        assert engine.is_file() and engine.stat().st_size, f"Missing engine: {engine}"

### 2A.2 Load the TensorRT Python bindings matching the station

The MS8 image ships TensorRT **libraries** and `trtexec`, but may not ship the TensorRT
**Python bindings**. Bindings must match the installed `libnvinfer8` exactly: a newer
PyPI `tensorrt` wheel would bring different libraries and could not load these engines.

If `import tensorrt` fails, the next cell asks APT for `python3-libnvinfer` **at exactly
the installed `libnvinfer8` version** from NVIDIA's configured Jetson repository. It
does not install anything system-wide (no `sudo`, no `pip`): it downloads the `.deb`,
checks its SHA256 against the APT index and its exact-version dependencies against the
installed packages, then extracts it under this benchmark's `tmp/` folder and adds it to
this kernel's `sys.path` only.

In [ ]:
if RUN_BENCHMARKS:
    import sys

    def dpkg_version(package):
        result = subprocess.run(["dpkg-query", "-W", "-f=${Version}", package],
                                capture_output=True, text=True)
        return result.stdout.strip() if result.returncode == 0 else None

    def unmet_exact_dependencies(depends, installed=dpkg_version):
        unmet = []
        for clause in depends.split(","):
            match = re.fullmatch(r"\s*([a-z0-9.+-]+)\s*\(\s*=\s*([^)\s]+)\s*\)\s*", clause)
            if match and installed(match[1]) != match[2]:
                unmet.append(f"{match[1]} (= {match[2]})")
        return unmet

    LIBNVINFER_VERSION = dpkg_version("libnvinfer8")
    assert LIBNVINFER_VERSION, "libnvinfer8 package not found"
    major, minor, patch = LIBNVINFER_VERSION.split(".")[:3]
    assert f"{int(major) * 1000 + int(minor) * 100 + int(patch)}" == TRT_VERSION, (
        f"libnvinfer8 {LIBNVINFER_VERSION} does not match trtexec v{TRT_VERSION}")
    TRT_PYTHON = {"libnvinfer8": LIBNVINFER_VERSION}
    try:
        import tensorrt as trt
        TRT_PYTHON["source"] = "preinstalled"
    except ModuleNotFoundError:
        package = f"python3-libnvinfer={LIBNVINFER_VERSION}"
        trt_python_root = BENCH_ROOT / "tensorrt-python"
        trt_python_root.mkdir()
        run(["apt-get", "download", package], cwd=trt_python_root)
        (deb,) = trt_python_root.glob("python3-libnvinfer_*.deb")
        index = subprocess.run(["apt-cache", "show", package], capture_output=True,
                               text=True, check=True).stdout
        assert f"SHA256: {sha256(deb)}" in index.splitlines(), "package hash differs from APT index"
        fields = dict(line.split(": ", 1) for line in subprocess.run(
            ["dpkg-deb", "-f", str(deb), "Package", "Version", "Architecture", "Depends"],
            capture_output=True, text=True, check=True).stdout.splitlines())
        assert fields["Version"] == LIBNVINFER_VERSION, fields
        unmet = unmet_exact_dependencies(fields["Depends"])
        assert not unmet, f"Installed TensorRT packages do not match: {unmet}"
        run(["dpkg-deb", "-x", str(deb), str(trt_python_root / "root")])
        site = (trt_python_root / "root" / "usr" / "lib" /
                f"python{sys.version_info.major}.{sys.version_info.minor}" / "dist-packages")
        assert (site / "tensorrt").is_dir(), f"bindings not built for this Python: {site}"
        sys.path.insert(0, str(site))
        import tensorrt as trt
        TRT_PYTHON.update(source=f"{deb.name} (extracted, not installed)", deb_sha256=sha256(deb))
    assert trt.__version__ == f"{major}.{minor}.{patch}", (trt.__version__, LIBNVINFER_VERSION)
    TRT_PYTHON["tensorrt_python"] = trt.__version__
    print(f"TensorRT Python {trt.__version__} from {trt.__file__}")

### 2A.3 Measure every backend with one timing method

This follows the inference timing used by [Ultralytics's predictor](https://github.com/ultralytics/ultralytics/blob/80c34c050921abd29970d8e3d2a2a41ec6538c98/ultralytics/engine/predictor.py#L380-L397):
[a synchronized wall-clock profiler](https://github.com/ultralytics/ultralytics/blob/80c34c050921abd29970d8e3d2a2a41ec6538c98/ultralytics/utils/ops.py#L20-L76)
around the inference call, identical for every backend. For **each request**:

```
cudaDeviceSynchronize() → start = perf_counter_ns() → one inference call → cudaDeviceSynchronize() → stop
```

| Same for all three variants | Value |
|---|---|
| Graph and input | NMS-free graph; one seeded FP32 input per batch, uploaded once and **shared** |
| I/O | Preallocated GPU buffers (no host↔device copy inside the timing) |
| Batches | 1, 4, 8 (TensorRT profile 1/8/8) |
| Requests | One request at a time (latency, not pipelined throughput) |
| Warm-up / samples | 20 warm-up and 50 timed requests per round, 3 rounds |
| Order | Backend order shuffled every round, so drift/thermal effects are shared |

The timed interval contains each library's Python call, its GPU work and completion,
i.e. the latency an application calling that backend observes. Reported: **median**,
p95 and mean per batch, ms per image, serial images/s (= batch × 1000 / median) and
speed-up versus ONNX Runtime at the same batch. These are raw-model figures, **not**
camera/pipeline FPS: for application-level performance use `eais-tools check perf`.

Outputs are copied back **after** timing and compared with ONNX Runtime (mean/max absolute
difference). Random inputs check numerical agreement, **not detection accuracy**; validate
labelled images before changing deployment precision.

`trtexec --loadEngine=... --noDataTransfers` remains useful for GPU-only *kernel* time,
but its GPU Compute Time excludes host dispatch and it pipelines requests, so it must
not be put in the same table. The research behind this choice is summarized in the README.

Station clocks/power mode are **not** changed. GPU load and frequency are recorded
before and after: if other pipelines use the GPU, all variants share that contention, so
ratios remain informative but absolute latency is higher than on an idle station.

In [ ]:
if RUN_BENCHMARKS:
    import ctypes, gc, platform, time
    from IPython.display import Markdown, display

    BATCHES = (1, 4, 8)
    WARMUP_REQUESTS, MEASURED_REQUESTS, ROUNDS = 20, 50, 3
    OUTPUTS = {"raw_boxes": 4, "raw_scores": 3}
    ORT_LABEL = "ONNX Runtime CUDA FP32"
    assert min(WARMUP_REQUESTS, MEASURED_REQUESTS, ROUNDS) > 0

    cudart = ctypes.CDLL("libcudart.so.12")

    def cuda_check(code, call):
        if code != 0:
            raise RuntimeError(f"{call} failed with CUDA error {code}")

    def device_sync():
        cuda_check(cudart.cudaDeviceSynchronize(), "cudaDeviceSynchronize")

    def gpu_state():
        load = Path("/sys/devices/platform/gpu.0/load")
        frequency = Path("/sys/class/devfreq/17000000.gpu/cur_freq")
        return {
            "gpu_load_percent": int(load.read_text()) / 10 if load.exists() else None,
            "gpu_frequency_mhz": int(frequency.read_text()) / 1e6 if frequency.exists() else None,
        }

    def summarize(label, batch, rounds, reference_median=None):
        assert len(rounds) == ROUNDS and all(len(r) == MEASURED_REQUESTS for r in rounds)
        values = np.asarray(rounds, dtype=np.float64)
        assert np.isfinite(values).all() and (values > 0).all(), label
        median = float(np.median(values))
        return {
            "variant": label, "batch": batch, "median_ms": median,
            "p95_ms": float(np.percentile(values, 95)), "mean_ms": float(values.mean()),
            "round_medians_ms": [float(np.median(r)) for r in values],
            "ms_per_image": median / batch, "serial_images_per_second": batch * 1000 / median,
            "speedup_vs_onnxruntime": (reference_median / median) if reference_median else 1.0,
        }

    state_before = gpu_state()
    if (state_before["gpu_load_percent"] or 0) > 10:
        print(f"WARNING: GPU already {state_before['gpu_load_percent']:.0f}% busy before the "
              "benchmark; absolute latencies will include that contention.")

    session = ort.InferenceSession(
        str(RAW_ONNX), providers=[("CUDAExecutionProvider", {"device_id": 0, "use_tf32": 0})])
    session.disable_fallback()
    providers = session.get_providers()
    assert providers[0] == "CUDAExecutionProvider", providers
    assert "TensorrtExecutionProvider" not in providers, providers
    ort_cuda_options = session.get_provider_options()["CUDAExecutionProvider"]
    assert ort_cuda_options["use_tf32"] == "0", ort_cuda_options

    trt_runtime = trt.Runtime(trt.Logger(trt.Logger.WARNING))
    contexts = {}
    for label, path in ENGINES.items():
        engine = trt_runtime.deserialize_cuda_engine(path.read_bytes())
        assert engine is not None, f"Cannot load {path}"
        io = {engine.get_tensor_name(i): (engine.get_tensor_mode(engine.get_tensor_name(i)),
                                          engine.get_tensor_dtype(engine.get_tensor_name(i)))
              for i in range(engine.num_io_tensors)}
        assert io == {"images": (trt.TensorIOMode.INPUT, trt.DataType.FLOAT),
                      **{name: (trt.TensorIOMode.OUTPUT, trt.DataType.FLOAT) for name in OUTPUTS}}, io
        contexts[label] = (engine, engine.create_execution_context())
    stream = ctypes.c_void_p()
    cuda_check(cudart.cudaStreamCreateWithFlags(ctypes.byref(stream), 1), "cudaStreamCreateWithFlags")

    def gpu_outputs(batch):
        return {name: ort.OrtValue.ortvalue_from_numpy(
                    np.zeros((batch, 25200, width), np.float32), "cuda", 0)
                for name, width in OUTPUTS.items()}

    order_rng = np.random.default_rng(2026)
    results, differences, orders = [], [], []
    for batch in BATCHES:
        host_input = np.random.default_rng(42).random((batch, 3, 640, 640), dtype=np.float32)
        gpu_input = ort.OrtValue.ortvalue_from_numpy(host_input, "cuda", 0)
        outputs = {ORT_LABEL: gpu_outputs(batch)}
        binding = session.io_binding()
        binding.bind_ortvalue_input("images", gpu_input)
        for name, value in outputs[ORT_LABEL].items():
            binding.bind_ortvalue_output(name, value)
        runners = {ORT_LABEL: lambda binding=binding: session.run_with_iobinding(binding)}
        for label, (engine, context) in contexts.items():
            outputs[label] = gpu_outputs(batch)
            assert context.set_input_shape("images", host_input.shape)
            assert context.set_tensor_address("images", gpu_input.data_ptr())
            for name, value in outputs[label].items():
                assert tuple(context.get_tensor_shape(name)) == tuple(value.shape()), name
                assert context.set_tensor_address(name, value.data_ptr())
            assert not context.infer_shapes()

            def run_engine(context=context):
                if not context.execute_async_v3(stream.value):
                    raise RuntimeError("TensorRT enqueue failed")
            runners[label] = run_engine

        samples = {label: [] for label in runners}
        for _ in range(ROUNDS):
            order = list(runners)
            order_rng.shuffle(order)
            orders.append({"batch": batch, "order": order})
            for label in order:
                for _ in range(WARMUP_REQUESTS):
                    runners[label]()
                timings = []
                for _ in range(MEASURED_REQUESTS):
                    device_sync()
                    start = time.perf_counter_ns()
                    runners[label]()
                    device_sync()
                    timings.append((time.perf_counter_ns() - start) / 1e6)
                samples[label].append(timings)

        reference = {name: value.numpy() for name, value in outputs[ORT_LABEL].items()}
        for label in runners:
            for name, width in OUTPUTS.items():
                values = outputs[label][name].numpy()
                assert values.shape == (batch, 25200, width) and np.isfinite(values).all(), (label, name)
                if label != ORT_LABEL:
                    error = np.abs(values - reference[name])
                    differences.append({
                        "variant": label, "batch": batch, "output": name,
                        "mean_absolute_difference": float(error.mean()),
                        "max_absolute_difference": float(error.max())})
        ort_row = summarize(ORT_LABEL, batch, samples[ORT_LABEL])
        results.append(ort_row)
        for label in contexts:
            results.append(summarize(label, batch, samples[label], ort_row["median_ms"]))
        for row in results[-len(runners):]:
            print(f"{row['variant']:<24} batch {batch}: median {row['median_ms']:8.3f} ms, "
                  f"p95 {row['p95_ms']:8.3f} ms", flush=True)
        del binding, runners, outputs, gpu_input
    state_after = gpu_state()
    cuda_check(cudart.cudaStreamDestroy(stream), "cudaStreamDestroy")
    del contexts, session
    gc.collect()

    report = {
        "utc": datetime.now(timezone.utc).isoformat(),
        "method": "per request: cudaDeviceSynchronize, perf_counter_ns, one inference call, "
                  "cudaDeviceSynchronize; shared GPU-resident input; preallocated GPU outputs",
        "source_onnx_sha256": sha256(source_onnx), "raw_onnx_sha256": sha256(RAW_ONNX),
        "engine_sha256": {label: sha256(engine) for label, engine in ENGINES.items()},
        "versions": {"python": platform.python_version(), "numpy": np.__version__,
                     "onnx": onnx.__version__, "onnxruntime": ort.__version__,
                     "trtexec": TRT_VERSION, **TRT_PYTHON},
        "l4t": " ".join(s.strip() for s in L4T),
        "onnxruntime_providers": providers, "onnxruntime_cuda_use_tf32": ort_cuda_options["use_tf32"],
        "tensorrt_build_flags": {"FP32": "--noTF32", "FP16": "--fp16 --noTF32"},
        "workspace_mib": WORKSPACE_MIB, "profile_min_opt_max_batch": [1, 8, 8],
        "warmup_requests": WARMUP_REQUESTS, "measured_requests_per_round": MEASURED_REQUESTS,
        "rounds": ROUNDS, "round_orders": orders,
        "gpu_state_before": state_before, "gpu_state_after": state_after,
        "results": results, "output_differences": differences,
        "deployment_engine_sha256": sha256(ENGINE_PATH),
        "note": "same timing boundary for all variants; raw NMS-free model latency, not pipeline FPS",
    }
    (BENCH_ROOT / "results.json").write_text(json.dumps(report, indent=2) + "\n")
    table = ("| Variant | Batch | Median ms | p95 ms | Round medians ms | ms/image | "
             "Serial images/s | Speed-up vs ORT |\n|---|---:|---:|---:|---:|---:|---:|---:|\n")
    for row in results:
        rounds = row["round_medians_ms"]
        table += (f"| {row['variant']} | {row['batch']} | {row['median_ms']:.2f} | "
                  f"{row['p95_ms']:.2f} | {min(rounds):.2f}-{max(rounds):.2f} | "
                  f"{row['ms_per_image']:.2f} | {row['serial_images_per_second']:.1f} | "
                  f"{row['speedup_vs_onnxruntime']:.2f}x |\n")
    display(Markdown(table + f"\nGPU before: {state_before}; after: {state_after}. "
                     "Raw model latency, one request at a time; not pipeline FPS."))
    print("Output differences versus ONNX Runtime (not an accuracy score):")
    for d in differences:
        print(f"  {d['variant']:<14} b{d['batch']} {d['output']:<10} "
              f"mean {d['mean_absolute_difference']:.3g} max {d['max_absolute_difference']:.3g}")
    print(f"Saved results and logs: {BENCH_ROOT}")

## 3. Validate and package with eais-tools

Use **`eais-tools aip check`, `aip build --include-file .include`, and `aip export`**.
EAIS Tools 0.9.2 copies all of `configs/` and each model's `1/`, selected model-root files,
root Markdown documents, manifest/callback/options, and `resources/logo.png`. It does NOT
automatically exclude every source file: an ONNX file inside `1/` would be copied.
`.include` adds our licenses, documentation, Node-RED flow and dashboard. Directory entries
are recursive and ignore `.gitignore`; `!` exclusion rules are not supported. This repository
therefore lists individual files. Missing entries, unexpected files and changed bytes fail
the checks below; nothing is silently stripped from a completed build.

The first cell uses the same EAIS config validators as the CLI, without launching a server.
Triton validation targets the deployment `config.pbtxt`, not old experiment configs in `tmp/`.
NvDCF uses OpenCV-style YAML; its header is normalized for a basic YAML/manifest/path check.
**EAIS `configs ds check` does not validate or run `nvtracker` in 0.9.2.** Actual tracker
execution is part of the full EAIS pipeline test in step 5, not a result of these static checks.

In [ ]:
import json, yaml
from eais_tools.triton_config_validator import validate_specified_triton_config
from eais_tools.ds_config_validator import validate_ds_config
from eais_tools.services.configs_service import validate_model_config

run(["eais-tools", "aip", "check", "--uuid", AIP_ID])
manifest = json.loads((AIP_ROOT / "manifest.json").read_text())
primary = manifest["primary_processor"]
model_config = AIP_ROOT / "models" / primary["name"] / "config.pbtxt"
triton_check = validate_specified_triton_config(str(model_config))
assert triton_check.result, triton_check.error_message
assert validate_model_config(primary["name"], str(model_config)), "Triton model/file mismatch"
ds_check = validate_ds_config(str(AIP_ROOT / "configs" / primary["config_plan"]))
assert ds_check.success, ds_check.message
tracker = manifest["tracker_engine"]
tracker_text = (AIP_ROOT / "configs" / tracker["config_plan"]).read_text()
tracker_config = yaml.safe_load(tracker_text.removeprefix("%YAML:1.0"))
assert isinstance(tracker_config, dict), "tracker YAML is not a mapping"
assert {"BaseConfig", "TargetManagement", "StateEstimator", "VisualTracker"} <= tracker_config.keys()
assert Path(tracker["lib_path"]).is_file(), tracker["lib_path"]
assert tracker["tracker_width"] > 0 and tracker["tracker_height"] > 0
print("OK: EAIS static config checks and tracker file checks; GPU runtime not yet tested")

In [ ]:
from eais_tools.include_parser import resolve_include_file

include_entries = [line.split("#", 1)[0].strip()
                   for line in (AIP_ROOT / ".include").read_text().splitlines()]
include_entries = [line for line in include_entries if line]
assert include_entries, ".include must not be empty"
for name in include_entries:
    path = Path(name)
    assert not path.is_absolute() and ".." not in path.parts, name
    assert not any(char in name for char in "!*?[]"), f"use explicit files in .include: {name}"
    assert (AIP_ROOT / path).is_file(), f"missing included file: {name}"
included = set(resolve_include_file(str(AIP_ROOT), include_filename=".include"))
assert included == set(include_entries), "EAIS include resolution differs from the explicit allowlist"
expected_files = included | {
    "manifest.json", "callback.py", "README.md", "CHANGELOG.md", "DESIGN.md", "PRODUCT.md",
    "resources/logo.png",
    "configs/config_infer_primary_detector_yolov7.txt", "configs/config_tracker_NvDCF_perf.yml",
    "models/AIPV1EMI0001_Yolo_Detector/config.pbtxt",
    "models/AIPV1EMI0001_Yolo_Detector/labels-rev3.txt",
    "models/AIPV1EMI0001_Yolo_Detector/libnvds_infercustomparser.so",
    "models/AIPV1EMI0001_Yolo_Detector/1/yolov7-fp16-1x8x8.engine",
}
if (AIP_ROOT / "options.json").is_file():
    expected_files.add("options.json")
for name in expected_files:
    path = AIP_ROOT / name
    assert path.is_file() and path.stat().st_size, f"missing/empty package source: {name}"
    assert not path.is_symlink() and not any(
        (AIP_ROOT / parent).is_symlink() for parent in Path(name).parents), name
    assert path.resolve().is_relative_to(AIP_ROOT.resolve()), f"external package source: {name}"
expected_hashes = {name: sha256(AIP_ROOT / name) for name in expected_files}
DIST = AIP_ROOT / "dist"
run(["eais-tools", "aip", "build", "--uuid", AIP_ID, "--include-file", ".include"])

In [ ]:
DIST = AIP_ROOT / "dist"
dist_paths = list(DIST.rglob("*"))
assert not any(p.is_symlink() for p in dist_paths), "symlinks must not ship"
actual_files = {p.relative_to(DIST).as_posix() for p in dist_paths if p.is_file()}
assert actual_files == set(expected_hashes), (
    f"dist/ mismatch: missing={sorted(set(expected_hashes) - actual_files)}, "
    f"unexpected={sorted(actual_files - set(expected_hashes))}")
for name, expected in expected_hashes.items():
    assert sha256(DIST / name) == expected, f"dist/ bytes differ from package source: {name}"

for p in sorted(DIST.rglob("*")):
    if p.is_file():
        print(p.relative_to(DIST))
print("\nOK: dist/ contains the runtime files and nothing else")

In [ ]:
import json, tarfile

assert not any(p.is_symlink() for p in DIST.rglob("*")), "symlinks must not ship"
assert {p.relative_to(DIST).as_posix(): sha256(p)
        for p in DIST.rglob("*") if p.is_file()} == expected_hashes, "dist/ changed since validation"
ARCHIVE = AIP_ROOT / f"{AIP_ID}_dist.tgz"
ARCHIVE.unlink(missing_ok=True)
run(["eais-tools", "aip", "export", "--uuid", AIP_ID])
assert ARCHIVE.is_file() and ARCHIVE.stat().st_size, "export did not create a package"
with tarfile.open(ARCHIVE, "r:gz") as archive:
    members = {}
    for member in archive.getmembers():
        path = Path(member.name)
        assert not path.is_absolute() and ".." not in path.parts, member.name
        assert member.isfile() or member.isdir(), f"unexpected archive entry: {member.name}"
        if member.isfile():
            name = path.relative_to("dist").as_posix()
            assert name not in members, f"duplicate archive entry: {member.name}"
            members[name] = member
    dist_files = {p.relative_to(DIST).as_posix(): p for p in DIST.rglob("*") if p.is_file()}
    assert set(members) == set(dist_files) == set(expected_hashes), "archive file set differs from validated dist/"
    for name, path in dist_files.items():
        with archive.extractfile(members[name]) as source:
            assert hashlib.sha256(source.read()).hexdigest() == sha256(path) == expected_hashes[name], name
    with archive.extractfile(members["manifest.json"]) as source:
        packaged_manifest = json.load(source)
    assert packaged_manifest == json.loads((AIP_ROOT / "manifest.json").read_text())
print(f"OK: verified package {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
print(f"SHA256: {sha256(ARCHIVE)}")

## 4. Register the AI processor

The AIP folder is already in the station's `/data/aip`, so instead of uploading an archive we ask
the video-streaming service (ms3) to **scan** the folder and then **register** the AIP. The upload
API (`POST /aiprocessor/upload`) is for stations in normal mode and is rejected in development
mode.

In [ ]:
import requests

MS3 = "http://ms3_videostreaming:8000"

requests.post(f"{MS3}/aiprocessor/scan", timeout=60).raise_for_status()
response = requests.get(f"{MS3}/aiprocessors", timeout=30)
response.raise_for_status()
aip = next(a for a in response.json() if a["unique_id"] == AIP_ID)
assert aip["version"] == packaged_manifest["version"], "ms3 discovered a different AIP version"
print(f"Found {aip['unique_id']} v{aip['version']} (pk {aip['pk']})")

r = requests.post(f"{MS3}/aiprocessor/{aip['pk']}/register", timeout=300)
r.raise_for_status()
print(r.json())
status = requests.get(f"{MS3}/aiprocessor/{aip['pk']}/status", timeout=30)
status.raise_for_status()
print(status.json())
print("Registration was requested; step 5 verifies that inference actually starts.")

## 5. Test a pipeline and check the callback output

**The runtime acceptance test is `eais-tools pipeline test`**, not a separate Triton server
or a hand-built `gst-launch` diagnostic in Jupyter. It tests the station-managed DeepStream
pipeline with this AIP's inference, NvDCF tracker and callback, with display output disabled.
No graphical session is required.

The tool uploads a sample video, creates `eais_pipeline`, starts it, and records callback
messages. It replaces existing `eais_pipeline`/`eais_media_*` resources before starting,
and cleanup removes every `eais_*` resource. The cell refuses to run while any `eais_*`
pipeline or media already exists. The tool listens until interrupted, so the
cell stops it with `SIGINT` after `LISTEN_SECONDS` (exit code 124 is expected and handled). `--retain_resources`
keeps the pipeline running so the dashboard in step 6 has live data. Step 7 removes it.
**A timeout or successful start request is not a pass**: subsequent cells require detected
objects in valid callback messages and identify exactly one newly created test pipeline.

The next cell then checks the messages against the contract documented in
[`doc/USER_MANUAL.md`](doc/USER_MANUAL.md#callback-output-metadata-contract): batches of
`{since, until, data}` records, where `data` maps each source ID to per-class statistics.

In [ ]:
import requests

MS3 = "http://ms3_videostreaming:8000"

def station_list(endpoint):
    response = requests.get(f"{MS3}/{endpoint}", timeout=30)
    response.raise_for_status()
    return response.json()

pipelines_before = station_list("pipelines")
media_before = station_list("medias")
conflicts = [p["pk"] for p in pipelines_before if p["name"].startswith("eais_")]
conflicts += [m["pk"] for m in media_before if m["name"].startswith("eais_")]
assert not conflicts, f"EAIS test would replace existing resources: {conflicts}; resolve ownership first"
PIPELINES_BEFORE = {p["pk"] for p in pipelines_before}
MEDIA_BEFORE = {m["pk"] for m in media_before}

DUMP = AIP_ROOT / "tmp" / "callback_dump.json"
DUMP.parent.mkdir(exist_ok=True)
DUMP.unlink(missing_ok=True)
LISTEN_SECONDS = 90
run(["timeout", "--signal=INT", "--kill-after=30", str(LISTEN_SECONDS),
     "eais-tools", "pipeline", "test", "--aip", AIP_ID, "--inputs_num", "1",
     "--callback_output", "--retain_resources", "--dump_file", str(DUMP)],
    env={**os.environ, "COLUMNS": "160"}, expected=(0, 124))

In [ ]:
import json

CLASSES = ["car", "person", "truck"]

# The dump file is a sequence of JSON arrays (one per received message), not a single document.
text, pos, records = DUMP.read_text(), 0, []
decoder = json.JSONDecoder()
while pos < len(text):
    if text[pos].isspace():
        pos += 1
        continue
    batch, pos = decoder.raw_decode(text, pos)
    records.extend(batch)
assert records, "no callback messages were received"

peak = {c: 0 for c in CLASSES}
for record in records:
    assert {"since", "until", "data"} <= record.keys(), record.keys()
    for source in record["data"].values():
        assert set(source["obj_counter"]) == set(CLASSES), source["obj_counter"].keys()
        assert {"objInROIcnt", "objLCCurrCnt"} <= source["analytics_obj_cnt"].keys()
        for c in CLASSES:
            peak[c] = max(peak[c], source["obj_counter"][c]["max_val"])

print(f"{len(records)} merge windows received, peak counts per frame: {peak}")
assert any(peak.values()), "no objects detected in the sample video"
print("OK: station-managed pipeline delivered detections and valid callback metadata")

In [ ]:
pipelines = [p for p in station_list("pipelines") if p["name"] == "eais_pipeline"]
assert len(pipelines) == 1, "expected exactly one retained EAIS test pipeline"
test_pipeline = pipelines[0]
assert test_pipeline["pk"] not in PIPELINES_BEFORE, "test pipeline was not created by this run"
PIPELINE_ID = test_pipeline["pk"]
TEST_MEDIA_IDS = {m["pk"] for m in station_list("medias")
                  if m["name"].startswith("eais_media_") and m["pk"] not in MEDIA_BEFORE}
assert len(TEST_MEDIA_IDS) == 1, "expected exactly one new EAIS sample video"
print(f"Test pipeline: {test_pipeline['name']} (pk {PIPELINE_ID})")

## 6. Install the Node-RED flow and dashboard

The repository ships a Node-RED project (`logic/`) and its UIBuilder dashboard (`dashboard/`).
This step installs them as the Node-RED project **`nodered-aipv1emi0001-dashboard`** in
`/data/projects`, connected to the test pipeline from step 5:

1. copy `logic/` to the project folder and `dashboard/` to its `uibuilder/` folder;
2. set the `AIP_PIPELINE_ID` environment variable of the **AIPV1EMI0001 Logic** tab;
3. install the dashboard's browser libraries (Vue, hls.js) with `npm ci`, so the page works
   without internet access afterwards (this step itself needs access to the npm registry);
4. make the folder a git repository, which Node-RED requires for projects.

The Node-RED nodes themselves (`@edge-ai/node-red-contrib-eais`, `node-red-contrib-uibuilder`)
are already installed on the station.

This step **does not switch** the station's active Node-RED project. Set `OVERWRITE = True` to
replace an earlier install; `flows_cred.json` is never copied.

In [ ]:
NR_PROJECT = "nodered-aipv1emi0001-dashboard"
OVERWRITE = False

project = Path("/data/projects") / NR_PROJECT
assert project.parent.is_dir(), "/data/projects is not mounted in this container"
if project.exists():
    assert OVERWRITE, f"{project} already exists; set OVERWRITE = True to replace it"
    shutil.rmtree(project)

project.mkdir()
for name in ("flows.json", "package.json", "package-lock.json", "README.md"):
    shutil.copy2(AIP_ROOT / "logic" / name, project / name)
shutil.copytree(AIP_ROOT / "dashboard", project / "uibuilder",
                ignore=shutil.ignore_patterns("node_modules", "README.md"))
(project / ".gitignore").write_text("node_modules/\nflows_cred.json\n.flows*.backup\n")

flows = json.loads((project / "flows.json").read_text())
logic_tab = next(n for n in flows if n["type"] == "tab" and n["label"] == "AIPV1EMI0001 Logic")
logic_tab["env"] = [{"name": "AIP_PIPELINE_ID", "value": PIPELINE_ID, "type": "str"}]
(project / "flows.json").write_text(json.dumps(flows, indent=4) + "\n")

subprocess.run(["npm", "ci", "--omit=dev", "--no-audit", "--no-fund"], cwd=project / "uibuilder", check=True)
git = ["git", "-c", "user.name=EdgeAI Station", "-c", "user.email=edgeai@localhost"]
subprocess.run(["git", "init", "-q", "-b", "main"], cwd=project, check=True)
subprocess.run(["git", "add", "-A"], cwd=project, check=True)
subprocess.run(git + ["commit", "-q", "-m", f"Install {AIP_ID} dashboard"], cwd=project, check=True)

for lib in ("vue", "hls.js"):
    assert (project / "uibuilder" / "node_modules" / lib / "package.json").is_file(), f"{lib} not installed"
print(f"OK: Node-RED project installed in {project}, wired to pipeline {PIPELINE_ID}")

### Open the project in Node-RED

These steps are manual on purpose, because opening a project replaces the flows currently running
in Node-RED for every user of the station.

1. Open the Node-RED editor from the station web UI.
2. Menu (☰) → **Projects** → **Open** → `nodered-aipv1emi0001-dashboard`.
3. Open the dashboard at `https://<station>/ms5/aipv1emi0001_dash/`. Counts for `car`, `person` and
   `truck` update every second, and the video plays from the pipeline's HLS stream.
4. To give other station users access, assign the `aipv1emi0001_dash` dashboard to them in
   **Accounts management**.

To follow another pipeline later, change `AIP_PIPELINE_ID` in the **AIPV1EMI0001 Logic** tab's
properties and deploy. If the callback node or ms3 is upgraded, enable the disabled
**inspect complete callback msg** debug node and compare a live message with the user manual.

## 7. Cleanup (keep the distributable package)

`pipeline clean` stops and deletes **all** `eais_*` pipelines and media. The cell
therefore checks that matching resources belong only to this notebook run before calling it,
then verifies their removal. The dashboard stops receiving data until you point
`AIP_PIPELINE_ID` at another pipeline.

The verified `AIPV1EMI0001_dist.tgz` archive and `dist/` are kept. Download the archive from
Jupyter to install it on another compatible station. Only the downloaded ONNX and build timing
cache are removed. Rebuilding later downloads the ONNX again.

Do **not** delete `models/.../1/yolov7-fp16-1x8x8.engine` or `models/.../libnvds_infercustomparser.so`
while the AIP is registered: ms3 copies the AIP folder each time a pipeline starts, so later starts
would fail.

Section 2A's benchmark engines, `results.json` and logs are retained under
`tmp/inference-benchmark/` for inspection and are not included in the exported package.
After reviewing them, you may remove only the specific benchmark run directories you
created there to reclaim space. They are never used by the registered deployment model.

In [ ]:
cleanup_pipelines = {p["pk"] for p in station_list("pipelines") if p["name"].startswith("eais_")}
cleanup_media = {m["pk"] for m in station_list("medias") if m["name"].startswith("eais_")}
assert cleanup_pipelines <= {PIPELINE_ID}, "pipeline clean would delete another run's pipeline"
assert cleanup_media <= TEST_MEDIA_IDS, "pipeline clean would delete another run's media"
run(["eais-tools", "pipeline", "clean"])
assert not ({p["pk"] for p in station_list("pipelines")} & {PIPELINE_ID}), "test pipeline still exists"
assert not ({m["pk"] for m in station_list("medias")} & TEST_MEDIA_IDS), "test media still exists"
for path in (MODEL_DIR / "yolov7tiny-640.onnx", AIP_ROOT / "bin" / "timing.cache"):
    path.unlink(missing_ok=True)
assert ARCHIVE.is_file(), "the distributable package must remain after cleanup"
print(f"Package preserved: {ARCHIVE}\nSHA256: {sha256(ARCHIVE)}")